# 05 — Train Conditional VAE (Phase 6)

**Goal:** Train a Conditional VAE on ISIC 2018 Task 3 so that in Phase 8 we can sample synthetic minority-class images (DF, VASC, AKIEC) and use them in Phase 10 to augment the classifier from Phase 5.

**Reference course material:** `notes-12-vae.py` (encoder/decoder pattern, reparameterise, loss structure, training loop), `notes-6-code-2-earlystopping__1_.py` (EarlyStopping class), `notes12generativeAI.pdf` (CVAE concept).

**Key design choices** (documented in `src/models/cvae.py`):
- Conditioning: one-hot class label concatenated to both flattened encoder features AND the latent z before the decoder.
- Reconstruction loss: **MSE** on [0,1] images (not BCE — BCE is fragile on natural RGB).
- latent_dim = 128 (Fashion-MNIST in the notes uses 2; we have 224×224 RGB).
- Inputs are NOT ImageNet-normalised — they stay in [0,1] so the decoder's Sigmoid matches.

**Outputs** (written to `results/`, matching the Phase 5 convention):
- `results/checkpoints/cvae_best.pt`
- `results/logs/cvae_log.csv`
- `results/plots/cvae_curves.png`
- `results/plots/cvae_reconstructions.png`
- `results/plots/cvae_samples_per_class.png`

## 1. Setup, paths, seed

In [ ]:
import sys
from pathlib import Path
import csv
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

# Project root convention from PROJECT_STATUS.md
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import ISICDataset, CLASS_NAMES   # noqa: E402
from src.models.cvae import CVAE, cvae_loss        # noqa: E402

# Seed (project convention: 42 everywhere)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')

# Output paths
RESULTS = PROJECT_ROOT / 'results'
(RESULTS / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RESULTS / 'logs').mkdir(parents=True, exist_ok=True)
(RESULTS / 'plots').mkdir(parents=True, exist_ok=True)

CKPT_PATH = RESULTS / 'checkpoints' / 'cvae_best.pt'
LOG_PATH  = RESULTS / 'logs'        / 'cvae_log.csv'
print(f'Checkpoint -> {CKPT_PATH}')
print(f'Log        -> {LOG_PATH}')

## 2. Hyperparameters

Two things differ from Phase 5:
1. **No ImageNet normalization** — the decoder ends in Sigmoid and outputs in [0,1], so inputs must also be in [0,1].
2. **No class weighting in the loss** — we want the CVAE to learn each class's distribution faithfully. We compensate for class imbalance later, when sampling: in Phase 8 we'll request more samples from rare classes.

> **beta**: starting at 1.0. After the first run, check `kl` in the logs. If `kl` collapses to ~0 → posterior collapse, lower beta. If `kl` keeps climbing while `recon` stops improving → raise beta. This is the Beta-VAE knob from slide 58 of `notes12generativeAI.pdf`.

In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 32       # 224x224 RGB + 20M params — drop to 16 if OOM
LATENT_DIM  = 128
N_CLASSES   = len(CLASS_NAMES)   # 7
EPOCHS      = 60
LR          = 1e-4     # Adam default 1e-3 is too aggressive for VAE on 224 RGB
BETA        = 1.0      # KL weight (tune after first run)
PATIENCE    = 10       # EarlyStopping patience on val_loss
NUM_WORKERS = 4

print(f'image size : {IMG_SIZE}')
print(f'batch size : {BATCH_SIZE}')
print(f'latent dim : {LATENT_DIM}')
print(f'epochs     : {EPOCHS}')
print(f'lr         : {LR}')
print(f'beta (KL)  : {BETA}')
print(f'patience   : {PATIENCE}')

## 3. Data — VAE-friendly transforms

Same `ISICDataset` and same `splits.csv` as Phase 5, but a different transform — we want pixels in [0,1], not ImageNet-normalised. We also skip strong augmentation: the VAE is trying to learn the data distribution, not be invariant to it. We do allow horizontal flip (skin lesions have no canonical left/right).

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),     # [0, 1]
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

splits_csv = PROJECT_ROOT / 'data' / 'processed' / 'splits.csv'
images_dir = PROJECT_ROOT / 'data' / 'raw' / 'ISIC2018_Task3_Training_Input'

train_ds = ISICDataset(splits_csv, images_dir, split='train', transform=train_tf)
val_ds   = ISICDataset(splits_csv, images_dir, split='val',   transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'train batches : {len(train_loader)}   ({len(train_ds)} images)')
print(f'val   batches : {len(val_loader)}   ({len(val_ds)} images)')

# Sanity-check one batch — should be RGB in [0, 1] and class ids in [0, 7)
x, y = next(iter(train_loader))
print(f'\nbatch x : {tuple(x.shape)}   dtype={x.dtype}   '
      f'min={x.min():.3f}  max={x.max():.3f}')
print(f'batch y : {tuple(y.shape)}   dtype={y.dtype}   '
      f'unique={sorted(y.unique().tolist())}')

## 4. Model, optimizer, EarlyStopping

In [ ]:
class EarlyStopping:
    """Identical to notes-6-code-2-earlystopping__1_.py."""
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False

    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
            print(f'  validation improved to {val_loss:.4f}')
        else:
            self.no_improvement_count += 1

        if self.no_improvement_count >= self.patience:
            self.stop_training = True


model = CVAE(latent_dim=LATENT_DIM, n_classes=N_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
early_stopping = EarlyStopping(patience=PATIENCE)

n_params = sum(p.numel() for p in model.parameters())
print(f'CVAE parameters: {n_params:,}')

## 5. Training loop

Mirrors `train()` in `notes-12-vae.py` but:
- passes labels `y` to the model (the C in CVAE),
- writes per-epoch metrics to `results/logs/cvae_log.csv`,
- saves the best checkpoint by val_loss,
- triggers EarlyStopping on val_loss.

In [ ]:
history = {'epoch': [], 'loss': [], 'recon': [], 'kl': [],
           'val_loss': [], 'val_recon': [], 'val_kl': []}
best_val = float('inf')

with open(LOG_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'loss', 'recon', 'kl', 'val_loss', 'val_recon', 'val_kl'])

for epoch in range(1, EPOCHS + 1):
    # -------------------- train --------------------
    model.train()
    run = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0}
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        x_recon, z_mean, z_log_var = model(x, y)
        loss, recon_l, kl_l = cvae_loss(x, x_recon, z_mean, z_log_var, beta=BETA)
        loss.backward()
        optimizer.step()

        run['loss']  += loss.item()
        run['recon'] += recon_l.item()
        run['kl']    += kl_l.item()

    n = len(train_loader)
    train_loss  = run['loss']  / n
    train_recon = run['recon'] / n
    train_kl    = run['kl']    / n

    # -------------------- val --------------------
    model.eval()
    vrun = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0}
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            x_recon, z_mean, z_log_var = model(x, y)
            loss, recon_l, kl_l = cvae_loss(x, x_recon, z_mean, z_log_var, beta=BETA)
            vrun['loss']  += loss.item()
            vrun['recon'] += recon_l.item()
            vrun['kl']    += kl_l.item()

    m = len(val_loader)
    val_loss  = vrun['loss']  / m
    val_recon = vrun['recon'] / m
    val_kl    = vrun['kl']    / m

    # -------------------- log / save / early stop --------------------
    history['epoch'].append(epoch)
    history['loss'].append(train_loss);   history['recon'].append(train_recon);  history['kl'].append(train_kl)
    history['val_loss'].append(val_loss); history['val_recon'].append(val_recon); history['val_kl'].append(val_kl)

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_loss, train_recon, train_kl,
                                val_loss, val_recon, val_kl])

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'loss={train_loss:.1f}  recon={train_recon:.1f}  kl={train_kl:.2f}  |  '
          f'val_loss={val_loss:.1f}  val_recon={val_recon:.1f}  val_kl={val_kl:.2f}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'latent_dim': LATENT_DIM,
            'n_classes': N_CLASSES,
            'beta': BETA,
        }, CKPT_PATH)
        print(f'  -> saved best checkpoint ({CKPT_PATH.name})')

    early_stopping.check_early_stop(val_loss)
    if early_stopping.stop_training:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nTraining done. Best val_loss = {best_val:.4f}')

## 6. Plot training curves

In [ ]:
log = pd.read_csv(LOG_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(log['epoch'], log['loss'],     label='train')
axes[0].plot(log['epoch'], log['val_loss'], label='val')
axes[0].set_title('Total loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(log['epoch'], log['recon'],     label='train')
axes[1].plot(log['epoch'], log['val_recon'], label='val')
axes[1].set_title('Reconstruction loss (MSE)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(log['epoch'], log['kl'],     label='train')
axes[2].plot(log['epoch'], log['val_kl'], label='val')
axes[2].set_title('KL divergence'); axes[2].set_xlabel('epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'plots' / 'cvae_curves.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 7. Reconstructions (sanity)

If the model is learning anything, reconstructions should at least preserve the **rough shape and colour** of the lesion. They will be blurry — that's the well-known VAE blurriness, and is one motivation for also building the GAN in Phase 7.

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f})")

# Pick one image per class from the val set
found = {}
for x, y in val_loader:
    for i in range(x.size(0)):
        c = y[i].item()
        if c not in found:
            found[c] = x[i]
        if len(found) == N_CLASSES:
            break
    if len(found) == N_CLASSES:
        break

classes = sorted(found.keys())
xs = torch.stack([found[c] for c in classes]).to(DEVICE)
ys = torch.tensor(classes, device=DEVICE)

with torch.no_grad():
    xr, _, _ = model(xs, ys)

xs = xs.cpu().permute(0, 2, 3, 1).numpy()
xr = xr.cpu().permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(2, N_CLASSES, figsize=(2.2 * N_CLASSES, 5))
for i, c in enumerate(classes):
    axes[0, i].imshow(xs[i].clip(0, 1)); axes[0, i].set_title(CLASS_NAMES[c]); axes[0, i].axis('off')
    axes[1, i].imshow(xr[i].clip(0, 1)); axes[1, i].axis('off')
axes[0, 0].set_ylabel('original', fontsize=11)
axes[1, 0].set_ylabel('reconstructed', fontsize=11)
plt.suptitle('CVAE reconstructions — one per class', y=1.02)
plt.tight_layout()
out = RESULTS / 'plots' / 'cvae_reconstructions.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 8. Conditional sampling (the whole point of Phase 6)

Sample 6 images per class from the prior N(0, I). These are what Phase 8 will use to augment the rare classes.

In [ ]:
N_SAMPLES_PER_CLASS = 6

fig, axes = plt.subplots(N_CLASSES, N_SAMPLES_PER_CLASS,
                         figsize=(1.8 * N_SAMPLES_PER_CLASS, 1.8 * N_CLASSES))

for c in range(N_CLASSES):
    samples = model.sample(class_idx=c, n=N_SAMPLES_PER_CLASS, device=DEVICE)
    samples = samples.cpu().permute(0, 2, 3, 1).numpy()
    for j in range(N_SAMPLES_PER_CLASS):
        ax = axes[c, j]
        ax.imshow(samples[j].clip(0, 1))
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], fontsize=11, rotation=0, labelpad=30, va='center')

plt.suptitle('CVAE samples — 6 per class from prior N(0, I)', y=1.01)
plt.tight_layout()
out = RESULTS / 'plots' / 'cvae_samples_per_class.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 9. What to look at after running

1. **`val_recon` curve**: should drop steadily. If flat from epoch 1 → bug; if drops then explodes → lower LR.
2. **`val_kl` curve**: should rise from ~0 and stabilise. If it collapses to 0 → posterior collapse, **lower BETA** and retrain. If it dominates total loss → **raise BETA**.
3. **Reconstructions**: should preserve rough lesion shape and dominant colour. Blurry is expected (VAE limitation; GAN in Phase 7 will sharpen).
4. **Per-class samples**: each row should look at least loosely like the class. If all rows look identical → the conditioning isn't working; debug the one-hot wiring in `cvae.py`.

